Nell'installazione delle dipendenze c'è un warning che però non impedisce al notebook di essere eseguito correttamente

In [1]:
board_meeting = """
VERBALE DI RIUNIONE - BOARD OF DIRECTORS
Data: 10 Gennaio 2025
Partecipanti: CEO, CTO, CFO, Head of Sales
Oggetto: Approvazione Roadmap "Project Mercury" (Integrazione LLM)

Discussione:
Il board ha discusso l'implementazione della nuova architettura basata su LlamaIndex per l'automazione documentale. Il CTO ha illustrato i vantaggi in termini di efficienza (-40% tempo di analisi).

Decisioni Strategiche Prese:
1. Approvazione del budget di 200.000€ per il progetto "Mercury".
2. Partnership strategica con fornitore Cloud per l'hosting delle GPU.
3. Riorganizzazione del team Data Science: stop alle assunzioni esterne, focus su formazione interna.

Action Items:
1. CTO: Finalizzare la scelta del vendor Cloud entro il 30/01. Priorità: Alta.
2. HR: Avviare corso di formazione Python/LLM per analisti junior. Priorità: Media. Costo: 5.000€. Timeline: Q1 2025.
3. CFO: Rilasciare prima tranche di fondi entro venerdì.

Clima della riunione:
Molto costruttivo ed entusiasta, nonostante alcune preoccupazioni iniziali sui costi di infrastruttura sollevate dal CFO."""


financial_report = """
DATA TRUST SOLUTIONS - FINANCIAL PERFORMANCE REPORT
Period: Fiscal Year 2024
Prepared by: CFO Office

Overview:
Il 2024 si chiude con risultati solidi, trainati dall'espansione nel mercato asiatico. I margini operativi sono in leggera contrazione a causa dei forti investimenti in R&D per la nuova piattaforma AI.

Metriche Finanziarie Chiave:
- Fatturato Totale (Revenue): 45.2 M€ (Unit: Euro). Variazione: +12.5% rispetto al FY2023.
- EBITDA: 8.4 M€ (Unit: Euro). Variazione: -2.1% rispetto al FY2023.
- Costi Operativi (OpEx): 32.1 M€. Variazione: +18% (dovuto a nuove assunzioni Tech).
- Net Profit: 5.1 M€.

Analisi dei Rischi Finanziari:
L'aumento dell'inflazione potrebbe erodere il potere d'acquisto dei clienti nel Q1 2025. La volatilità del tasso di cambio EUR/USD rappresenta un rischio moderato per i contratti software internazionali.

Sentiment di Mercato:
Gli analisti mantengono un outlook "Positivo" sul titolo, apprezzando la strategia di lungo termine basata sull'AI, nonostante la contrazione dell'EBITDA a breve termine.

"""


report_compliance = """
DATA TRUST SOLUTIONS - INTERNAL AUDIT REPORT
CONFIDENTIAL
Date: 2024-12-15
Subject: GDPR Compliance Audit - Q4 2024

Executive Summary:
L'audit del quarto trimestre evidenzia un buon livello generale di conformità, tuttavia sono state rilevate criticità severe nella gestione dei dati biometrici. Il punteggio complessivo di conformità è stimato all'85%.

Dettagli Rischi e Non Conformità:
1. CRITICAL: I log di accesso al server "SecureCore" non sono criptati. In caso di attacco, i dati utenti sono esposti.
   - Area: IT Security
   - Impatto: Alto (Possibili sanzioni fino al 4% del fatturato)
   - Costo Mitigazione: 15.000€

2. MEDIUM: Mancata cancellazione dei CV dei candidati scartati dopo 24 mesi.
   - Area: HR Data Retention
   - Impatto: Medio (Rischio reputazionale)
   - Costo Mitigazione: 2.000€

Piano d'Azione Richiesto:
1. Implementare crittografia AES-256 sui log entro il 15/01/2025. Priorità: Alta. Owner: M. Rossi (CTO). Costo stimato: 15k€.
2. Aggiornare script di auto-cancellazione database HR entro fine mese. Priorità: Media. Owner: IT Support.

Conclusioni:
È urgente intervenire sulla crittografia. Il resto dei processi è conforme agli standard ISO 27001.

"""

In [8]:
with open("board_meeting_jan25.txt", "w") as f:
    f.write(board_meeting)

with open("financial_report_2024.txt", "w") as f:
    f.write(financial_report)

with open("report_compliance", "w") as f:
    f.write(report_compliance)

In [3]:
import sys

if 'google.colab' in sys.modules:
    !pip install -q -U google-genai

!pip install llama-index
!pip install langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.1/719.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.9/234.9 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.47.0 which is incompatible.
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/w

In [4]:
import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Optional
from datetime import datetime
import time

from google import genai
from google.colab import userdata
from llama_index.core import SimpleDirectoryReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
@dataclass
class DocumentMetadata:
    doc_id: str
    title: str
    doc_type: str
    date: str
    source: Optional[str]
    processed_at: str


@dataclass
class RiskAssessment:
    level: str
    area: str
    impact: str
    mitigation_cost: float = 0.0


@dataclass
class ActionItem:
    priority: str
    action: str
    timeline: str
    estimated_cost: float = 0.0
    owner: str = ""


@dataclass
class FinancialMetric:
    name: str
    value: str
    unit: str
    change_pct: float = 0.0
    comparison_period: str = ""


@dataclass
class DocumentInsights:
    metadata: DocumentMetadata
    executive_summary: str
    key_points: List[str]
    sentiment: str
    urgency: str
    risks: List[RiskAssessment]
    actions: List[ActionItem]
    financial_metrics: Optional[List[FinancialMetric]] = None
    strategic_decisions: Optional[List[Dict[str, Any]]] = None
    compliance_score: Optional[float] = None

In [6]:
class DocumentProcessor:

    def __init__(
        self,
            api_key: str,
        gemini_model_name: str = "gemini-2.5-flash",
        chunk_size: int = 3000,
        chunk_overlap: int = 400,
        min_chunk_size: int = 500
    ):
        self.gemini_model_name = gemini_model_name
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.min_chunk_size = min_chunk_size
        self.prompts = self._init_prompts()
        self.max_retries = 3

        # Initialize Gemini model
        self.client = genai.Client(api_key=api_key)

        # Inizializza text splitter semantico
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=[
                "\n\n",
                "\n",
                ". ",
                ", ",
                " ",
                ""
            ],
            is_separator_regex=False
        )

    def _init_prompts(self) -> Dict[str, str]:

        base_rules = """
        SEI UN API JSON. Rispondi SEMPRE ed ESCLUSIVAMENTE in lingua ITALIANA.
        Ogni valore testuale nel JSON deve essere in ITALIANO.
        Rispondi SOLO con un JSON valido. Nessun testo prima o dopo.
        Se un campo non è presente, usa null o [].
        """

        return {
            "compliance": base_rules + """
            Analizza questo report di conformità.
            JSON Schema:
            {
                "executive_summary": "Sintesi discorsiva in italiano",
                "key_points": ["Elenco punti chiave in italiano"],
                "sentiment": "Positivo/Negativo/Neutro",
                "urgency": "Alta/Media/Bassa",
                "risks": [{"level": "Critico/Alto/Medio", "area": "italiano", "impact": "descrizione in italiano", "mitigation_cost": float}],
                "actions": [{"priority": "Alta/Media", "action": "azione in italiano", "timeline": "scadenza", "estimated_cost": float, "owner": "responsabile"}],
                "compliance_score": float
            }
            TESTO DA ANALIZZARE: {text}
            """,

            "financial": base_rules + """
            Analizza questo report finanziario.
            JSON Schema:
            {
                "executive_summary": "Sintesi finanziaria in italiano",
                "key_points": ["Metriche principali"],
                "financial_metrics": [{"name": "nome metrica in italiano", "value": "valore", "unit": "valuta/unità", "change_pct": float, "comparison_period": "periodo"}],
                "risks": [{"level": "string", "area": "string", "impact": "descrizione in italiano"}],
                "sentiment": "Positivo/Neutro/Negativo",
                "urgency": "Bassa/Media/Alta"
            }
            TESTO DA ANALIZZARE: {text}
            """,

            "meeting": base_rules + """
            Analizza questo verbale di riunione.
            JSON Schema:
            {
                "executive_summary": "Riassunto della riunione in italiano",
                "key_points": ["Punti discussi"],
                "strategic_decisions": [{"decision": "decisione presa in italiano"}],
                "actions": [{"priority": "string", "action": "compito in italiano", "owner": "string", "timeline": "string", "estimated_cost": float}],
                "sentiment": "Costruttivo/Teso/Formale",
                "urgency": "string"
            }
            TESTO DA ANALIZZARE: {text}
            """,

            "aggregator": base_rules + """
            Unisci i seguenti riassunti parziali in un unico Executive Summary coerente e una lista di Key Points unica.
            TUTTO IL TESTO DEVE ESSERE IN ITALIANO.
            INPUT: {text}
            """
        }


    def load_document(self, file_path: str) -> List[str]:
        """
        Carica documento con LlamaIndex e applica chunking semantico.
        """
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f"File not found: {path}")

        print(f"Loading document: {path.name}")

        # LlamaIndex: gestisce PDF, Docx, Txt, Email automaticamente
        reader = SimpleDirectoryReader(input_files=[str(path)])
        documents = reader.load_data()

        # Unione testo da tutte le pagine
        full_text = "\n\n".join([d.text for d in documents])

        print(f"Document length: {len(full_text)} characters")

        # Chunking intelligente con RecursiveCharacterTextSplitter
        raw_chunks = self.text_splitter.split_text(full_text)

        # Filtra chunk troppo piccoli (rumore)
        valid_chunks = [c for c in raw_chunks if len(c.strip()) >= self.min_chunk_size]

        print(f"Created {len(valid_chunks)} semantic chunks (overlap: {self.chunk_overlap} chars)")

        return valid_chunks

    def _sanitize_json(self, text: str) -> str:
        """Pulisce l'output per estrarre solo il JSON valido"""
        # Rimuovi markdown code blocks
        text = re.sub(r'```json\s*|\s*```', '', text)
        text = text.strip()

        # Trova il primo { e l'ultimo }
        start = text.find('{')
        end = text.rfind('}')

        if start != -1 and end != -1:
            return text[start:end+1]

        return text

    def _call_llm(self, prompt_template: str, text: str, context: str = "") -> Dict[str, Any]:
        """
        Chiamata LLM con retry, validazione e gestione errori (adattata per Gemini).

        """
        # Limita lunghezza per evitare overflow
        max_text_len = 5000
        if len(text) > max_text_len:
            text = text[:max_text_len] + "\n[...troncato]"

        prompt = prompt_template.replace("{text}", text)
        raw_content = ""

        for attempt in range(self.max_retries):
            try:
                # Gemini API call
                response = self.client.models.generate_content(
                    model=self.gemini_model_name,
                    contents=prompt,
                    config={
                        "temperature": 0.15,
                        "top_p": 0.5,
                        "response_mime_type": "application/json"
                    }
                )

                raw_content = response.text
                clean_json = self._sanitize_json(raw_content)

                result = json.loads(clean_json)

                if not isinstance(result, dict):
                    raise ValueError("Output is not a dictionary")

                # Validazione campi obbligatori
                required_fields = ["executive_summary", "sentiment", "urgency"]
                for field in required_fields:
                    if field not in result:
                        result[field] = "Neutral" if field == "sentiment" else "Low"

                return result

            except json.JSONDecodeError as e:
                print(f"JSON decode error (attempt {attempt+1}/{self.max_retries}): {str(e)[:100]}")
                print(f"Raw content was: {raw_content[:200]}...")
                if attempt < self.max_retries - 1:
                    time.sleep(1)
                else:
                    print(f"Failed to parse JSON after {self.max_retries} attempts")
                    return {
                        "executive_summary": text[:300] + "...",
                        "key_points": ["Analysis failed - manual review required"],
                        "sentiment": "Neutral",
                        "urgency": "Low"
                    }

            except Exception as e:
                print(f"LLM error (attempt {attempt+1}/{self.max_retries}): {str(e)[:100]}")
                if attempt == self.max_retries - 1:
                    return {
                        "executive_summary": "Error during analysis",
                        "key_points": ["Processing error"],
                        "sentiment": "Neutral",
                        "urgency": "Low"
                    }

    def analyze(self, chunks: List[str], metadata: DocumentMetadata) -> DocumentInsights:
        """
        Analizza chunks con pattern Map-Reduce ottimizzato.

        Strategia:
        1. MAP: Analisi parallela dei chunk (con contesto overlap)
        2. REDUCE: Aggregazione intelligente dei risultati
        3. MERGE: Costruzione oggetto finale con deduplicazione
        """

        start_time = time.time()
        print(f"\nAnalyzing {len(chunks)} semantic chunk(s) for {metadata.title}...")

        partial_results = []
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i+1}/{len(chunks)} ({len(chunk)} chars)...")

            # Passa il prompt specifico per il tipo di documento
            res = self._call_llm(self.prompts[metadata.doc_type], chunk)
            partial_results.append(res)

        if len(chunks) > 1:

            summaries = []
            for i, r in enumerate(partial_results):
                summary = r.get("executive_summary", "")
                if summary and summary not in ["Error during analysis", ""]:
                    summaries.append(f"[Part {i+1}] {summary}")


            if summaries:
                agg_input = "\n\n".join(summaries)
                agg_res = self._call_llm(self.prompts["aggregator"], agg_input)

                final_summary = agg_res.get("executive_summary", summaries[0])
                final_keys = agg_res.get("key_points", [])
            else:
                final_summary = "Unable to generate comprehensive summary"
                final_keys = []
        else:
            final_summary = partial_results[0].get("executive_summary", "")
            final_keys = partial_results[0].get("key_points", [])


        def collect_list(key: str, cls=None) -> List:
            """Raccoglie, valida e deduplica liste da risultati parziali"""
            items = []
            seen = set()  # Per deduplicazione

            for r in partial_results:
                raw_list = r.get(key, [])
                if raw_list and isinstance(raw_list, list):
                    if cls:

                        valid_fields = cls.__annotations__.keys()
                        for item in raw_list:
                            if isinstance(item, dict):

                                clean_item = {k: v for k, v in item.items() if k in valid_fields}


                                for field, field_type in cls.__annotations__.items():
                                    if field not in clean_item:
                                        if field_type == float:
                                            clean_item[field] = 0.0
                                        elif field_type == str:
                                            clean_item[field] = ""


                                item_key = str(sorted(clean_item.items()))
                                if item_key not in seen:
                                    try:
                                        items.append(cls(**clean_item))
                                        seen.add(item_key)
                                    except Exception as e:
                                        print(f"Skipping invalid {cls.__name__}: {str(e)[:80]}")
                    else:

                        for item in raw_list:
                            if isinstance(item, str) and item not in seen:
                                items.append(item)
                                seen.add(item)
                            elif isinstance(item, dict):

                                item_str = json.dumps(item, sort_keys=True)
                                if item_str not in seen:
                                    items.append(item)
                                    seen.add(item_str)

            return items

        # Costruzione oggetto finale
        insights = DocumentInsights(
            metadata=metadata,
            executive_summary=final_summary,
            key_points=list(set(final_keys)) if final_keys else collect_list("key_points"),
            sentiment=self._aggregate_sentiment([r.get("sentiment", "Neutral") for r in partial_results]),
            urgency=self._aggregate_urgency([r.get("urgency", "Low") for r in partial_results]),
            risks=collect_list("risks", RiskAssessment),
            actions=collect_list("actions", ActionItem),
            financial_metrics=collect_list("financial_metrics", FinancialMetric) if metadata.doc_type == "financial" else None,
            strategic_decisions=collect_list("strategic_decisions") if metadata.doc_type == "meeting" else None,
            compliance_score=self._average_score([r.get("compliance_score") for r in partial_results])
        )

        elapsed = time.time() - start_time
        print(f"Analysis completed in {elapsed:.1f}s")

        return insights

    def _aggregate_sentiment(self, sentiments: List[str]) -> str:
        """Aggregazione sentiment con logica votazione"""
        if not sentiments:
            return "Neutral"

        # Mappa priorità
        priority = {"Negative": 3, "Neutral": 2, "Positive": 1}
        # Ritorna il sentiment più critico (se c'è almeno un Negative, prevale)
        return max(set(sentiments), key=lambda s: priority.get(s, 2))

    def _aggregate_urgency(self, urgencies: List[str]) -> str:
        """Aggregazione urgency con logica votazione"""
        if not urgencies:
            return "Low"

        priority = {"High": 3, "Medium": 2, "Low": 1}
        return max(set(urgencies), key=lambda u: priority.get(u, 1))

    def _average_score(self, scores: List[Optional[float]]) -> Optional[float]:
        """Calcola media di compliance score"""
        valid_scores = [s for s in scores if s is not None and isinstance(s, (int, float))]
        if not valid_scores:
            return None
        return round(sum(valid_scores) / len(valid_scores), 1)

In [10]:
if __name__ == "__main__":

    # Configure Gemini API
    GOOGLE_API_KEY=userdata.get('Gemini')


    # Configurazione
    processor = DocumentProcessor(
        api_key=GOOGLE_API_KEY,
        gemini_model_name="gemini-2.5-flash",
        chunk_size=3000,      # Chunk più grandi per contesto
        chunk_overlap=400,    # 13% overlap per continuità
        min_chunk_size=500    # Filtra chunk rumore
    )

    # Test sui 3 documenti
    test_cases = [
        ("/content/report_compliance", "compliance", "GDPR Audit Q4 2024"),
        ("/content/financial_report_2024.txt", "financial", "Performance Finanziaria FY2024"),
        ("/content/board_meeting_jan25.txt", "meeting", "Verbale Board 10/01/2025")
    ]

    results = []
    total_start = time.time()

    for file_path, doc_type, title in test_cases:
        print(f"\n{'-'*70}")
        print(f"Processing: {title}")
        print(f"{'-'*70}")

        metadata = DocumentMetadata(
            doc_id=f"{doc_type.upper()}-{datetime.now().strftime('%Y%m%d%H%M')}",
            title=title,
            doc_type=doc_type,
            date=datetime.now().strftime("%Y-%m-%d"),
            source="Internal Documents",
            processed_at=datetime.now().isoformat()
        )

        try:
            # Load & Chunk
            chunks = processor.load_document(file_path)

            # Analyze
            insights = processor.analyze(chunks, metadata)

            # Report
            print(f"\nResults Summary:")
            print(f"  • Summary: {insights.executive_summary[:120]}...")
            print(f"  • Key Points: {len(insights.key_points)}")
            print(f"  • Risks: {len(insights.risks)}")
            print(f"  • Actions: {len(insights.actions)}")
            print(f"  • Sentiment: {insights.sentiment}")
            print(f"  • Urgency: {insights.urgency}")

            if insights.compliance_score:
                print(f"  • Compliance Score: {insights.compliance_score}%")

            if insights.financial_metrics:
                print(f"  • Financial Metrics: {len(insights.financial_metrics)}")

            # Converto in dict per salvare
            result_dict = {
                "metadata": asdict(metadata),
                "executive_summary": insights.executive_summary,
                "key_points": insights.key_points,
                "sentiment": insights.sentiment,
                "urgency": insights.urgency,
                "risks": [asdict(r) for r in insights.risks],
                "actions": [asdict(a) for a in insights.actions],
                "financial_metrics": [asdict(m) for m in insights.financial_metrics] if insights.financial_metrics else None,
                "strategic_decisions": insights.strategic_decisions,
                "compliance_score": insights.compliance_score
            }
            results.append(result_dict)

        except Exception as e:
            print(f"Error processing {title}: {e}")
            import traceback
            traceback.print_exc()

    # Salva risultati
    output_path = Path("outputs/final_results.json")
    output_path.parent.mkdir(exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    total_time = time.time() - total_start

    print(f"\n{'-'*70}")
    print(f"Pipeline completata!")
    print(f"  • Total time: {total_time:.1f}s")
    print(f"  • Documents processed: {len(results)}")
    print(f"  • Results saved to: {output_path}")
    print(f"{'-'*70}")


----------------------------------------------------------------------
Processing: GDPR Audit Q4 2024
----------------------------------------------------------------------
Loading document: report_compliance
Document length: 1189 characters
Created 1 semantic chunks (overlap: 400 chars)

Analyzing 1 semantic chunk(s) for GDPR Audit Q4 2024...
Processing chunk 1/1 (1186 chars)...
Analysis completed in 7.8s

Results Summary:
  • Summary: L'audit del quarto trimestre evidenzia un buon livello generale di conformità, tuttavia sono state rilevate criticità se...
  • Key Points: 5
  • Risks: 2
  • Actions: 2
  • Sentiment: Negativo
  • Urgency: Alta
  • Compliance Score: 85.0%

----------------------------------------------------------------------
Processing: Performance Finanziaria FY2024
----------------------------------------------------------------------
Loading document: financial_report_2024.txt
Document length: 1044 characters
Created 1 semantic chunks (overlap: 400 chars)

Analyzi